In [1]:
import json
from pathlib import Path

import pandas as pd
import polars as pl
import yaml
import numpy as np
import structlog

from common import read_day, daterange

with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

days = []
for i in config["days"]:
    days += list(daterange(i["start"], i["end"]))

log = structlog.get_logger()

In [2]:
OUTPUT = f"{config['dirs']['output']}/{config['city']}/id_lookup"

In [20]:
def collect_ids(
    raw_pings_path: str, id_column: str, id_type_column: str
) -> dict[str, int]:
    ids = pl.DataFrame()
    for i in days:
        temp = read_day(i, raw_pings_path).with_columns(
            pl.concat_str(
                [pl.col("device_aid_hashed"), pl.col("device_aid_type")], separator="."
            ).alias("id_with_type")
        )
        temp = temp.unique(subset=["id_with_type"], keep="first")
        ids = pl.concat([ids, temp])
    ids = (
        ids.unique(subset=["id_with_type"], keep="first")
        .sort("id_with_type")
        .with_row_index("device_id", offset=1)
    )
    lookup = dict(zip(ids["id_with_type"], ids["device_id"]))
    return lookup


def create_partitions(lookup: dict[str, int]):
    return {v: k[:2] for k, v in lookup.items()}

In [22]:
try:
    with open(f"{OUTPUT}/id_lookup.json", "r") as fp:
        lookup = json.load(fp)
except FileNotFoundError:
    # runs 25 minutes for Budapest data
    lookup = collect_ids(
        config["dirs"]["raw_source"],
        id_column=config["id_column"],
        id_type_column=config["id_type_column"],
    )
    # log.info(f"number unique devices: {len(np.unique(lookup))}")

    Path(OUTPUT).mkdir(parents=True, exist_ok=True)
    with open(f"{OUTPUT}/id_lookup.json", "w") as fp:
        json.dump(lookup, fp, indent=1)

: 

## create partitions from original IDs

In [5]:
partition = create_partitions(lookup)

with open(f"{OUTPUT}/partition.json", "w") as fp:
    json.dump(partition, fp, indent=1)

### descriptives

In [11]:
partition_distribution = (
    pd.DataFrame({"partition": partition.values(), "device_id": partition.keys()})
    .groupby("partition")["device_id"]
    .count()
    .reset_index()
)
partition_distribution["device_id"].describe()

count    256.000000
mean     377.832031
std       19.991349
min      309.000000
25%      364.750000
50%      376.500000
75%      392.000000
max      424.000000
Name: device_id, dtype: float64